# Suffix Structures & Advanced String Matching

The **string algorithms** notebook matched a *single* pattern in a text (KMP, Rabin-Karp). This one tackles the other half: **preprocess a text once, then answer many substring/repeat queries fast** — and **match many patterns at once**. These are the workhorses behind search indexes, bioinformatics, plagiarism/dedup, and intrusion detection.

Four structures, each built from scratch and each *proven* against a brute-force reference over thousands of random strings:

- the **Z-array** (linear-time prefix matching, a cousin of **KMP**'s prefix function),
- the **suffix array** (all suffixes in sorted order),
- the **LCP array** (adjacent-suffix overlaps, via Kasai), and
- the **Aho-Corasick** automaton (a multi-pattern generalization of KMP built on a **trie**).

**Contents**

1. **The problem** — one text, many queries; or many patterns
2. **Z-algorithm** — the Z-array in O(n), and matching via `P + sep + T`
3. **Suffix array** — sorted suffixes by prefix-doubling
4. **LCP array** — Kasai's algorithm and the longest repeated substring
5. **Aho-Corasick** — many patterns in one O(|T|) scan
6. **When to use what**
7. **Complexity cheat-sheet**

In [1]:
import random
from collections import deque

rng = random.Random(20240602)          # seeded -> every 'random' check below is reproducible

def brute_search(text, pattern):       # naive find-all: our ground truth for single patterns
    m = len(pattern)
    if m == 0:
        return list(range(len(text) + 1))
    return [i for i in range(len(text) - m + 1) if text[i:i + m] == pattern]

print("brute_search('mississippi', 'ssi') ->", brute_search('mississippi', 'ssi'))

brute_search('mississippi', 'ssi') -> [2, 5]


## 1. The problem — one text, many queries; or many patterns

Single-pattern search (the **string algorithms** notebook) preprocesses the *pattern* and scans the *text*: great when the pattern is fixed. Two different shapes need different machinery:

| Shape | What's fixed | What changes | Tool |
|---|---|---|---|
| One pattern, one text | nothing | — | **KMP** / Rabin-Karp (string algorithms) |
| **One text, many queries** | the **text** | the queries (substring? count? longest repeat?) | **suffix array + LCP** |
| **Many patterns, one scan** | the **pattern set** | the text streams by | **Aho-Corasick** |

The unifying idea is **suffixes**. Every substring of a text is a *prefix of some suffix*, so a structure that organizes all `n` suffixes (a sorted array, or a tree) answers substring questions by binary-searching or walking that structure — the heavy lifting happens once, in preprocessing.

We start with the **Z-array**, the simplest of these tools and the gateway to the rest.

## 2. Z-algorithm — the Z-array in O(n)

For a string `s`, the **Z-array** is defined by

> `z[i]` = length of the longest substring starting at `i` that is also a **prefix** of `s`.

(`z[0]` is conventionally `n`, the whole string.) Computing each `z[i]` by a fresh comparison loop would be O(n²); the Z-algorithm does it in **O(n)** by maintaining the rightmost prefix-match interval `[l, r)` seen so far (the **Z-box**). When `i` falls inside that box, the value at the mirror position `i - l` gives a free head start — we only ever compare characters past `r`, so `r` advances at most `n` times total.

It's the same *reuse-what-you-learned* trick as **KMP**'s prefix function, expressed as prefix-match lengths instead of border lengths.

In [2]:
def z_array(s):
    n = len(s)
    z = [0] * n
    z[0] = n                                   # whole string matches itself
    l = r = 0                                  # [l, r) = rightmost prefix-match box so far
    for i in range(1, n):
        if i < r:
            z[i] = min(r - i, z[i - l])        # reuse the mirror value, capped at the box edge
        while i + z[i] < n and s[z[i]] == s[i + z[i]]:
            z[i] += 1                          # extend only past r -> O(n) amortized
        if i + z[i] > r:
            l, r = i, i + z[i]                 # found a box reaching further right
    return z

print("z_array('aabxaabxay'):", z_array('aabxaabxay'))
#                                ^ z[4]=5: 'aabxa' at index 4 re-matches the prefix 'aabxa'
print("z_array('mississippi'):", z_array('mississippi'))

z_array('aabxaabxay'): [10, 1, 0, 0, 5, 1, 0, 0, 1, 0]
z_array('mississippi'): [11, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


### Pattern matching via `P + sep + T`

To find a pattern `P` in a text `T`, concatenate `P + sep + T` where `sep` is a character in neither (here `"\x00"`). Any position `i` in the text part with `z[i] >= len(P)` is an exact occurrence — the suffix there shares a `|P|`-length prefix with the whole concatenation, i.e. with `P` itself. The separator stops a Z-box from ever spanning the boundary, so a match length can't exceed `|P|`.

In [3]:
def z_search(text, pattern, sep='\x00'):
    assert sep not in text and sep not in pattern, 'separator must be absent from both'
    m = len(pattern)
    if m == 0:
        return list(range(len(text) + 1))
    z = z_array(pattern + sep + text)
    # text char at concat index j corresponds to text position j - (m + 1)
    return [j - m - 1 for j in range(m + 1, len(z)) if z[j] >= m]

print("z_search('mississippi', 'ssi'):", z_search('mississippi', 'ssi'))
print("z_search('abababab', 'aba')   :", z_search('abababab', 'aba'))   # overlapping hits

# PROOF: Z definition matches its literal meaning, and z_search matches brute force.
def z_brute(s):                                # O(n^2) reference for the Z definition
    n = len(s); out = [0] * n; out[0] = n
    for i in range(1, n):
        k = 0
        while i + k < n and s[k] == s[i + k]:
            k += 1
        out[i] = k
    return out

for _ in range(3000):
    s = ''.join(rng.choice('ab') for _ in range(rng.randint(1, 14)))
    assert z_array(s) == z_brute(s)
    t = ''.join(rng.choice('ab') for _ in range(rng.randint(0, 16)))
    p = ''.join(rng.choice('ab') for _ in range(rng.randint(1, 4)))
    assert z_search(t, p) == brute_search(t, p)
print('Z-array == definition, and z_search == brute force: 3000 random cases OK')

z_search('mississippi', 'ssi'): [2, 5]
z_search('abababab', 'aba')   : [0, 2, 4]
Z-array == definition, and z_search == brute force: 3000 random cases OK


## 3. Suffix array — sorted suffixes by prefix-doubling

The **suffix array** of `s` is the list of starting indices of all `n` suffixes, **sorted by the suffixes themselves**. It's the memory-light cousin of a suffix tree: once you have it, *every substring* lives in a contiguous band (because substrings sharing a prefix sort together), so substring existence is a binary search.

The naive build — `sorted(range(n), key=lambda i: s[i:])` — is correct but **O(n² log n)**: each comparison copies and compares O(n)-length suffixes. **Prefix doubling** fixes the comparison cost. Rank suffixes first by their first character, then repeatedly *double* the compared length: a suffix's order by its first `2k` characters is `(rank_by_first_k, rank_of_the_suffix k positions later)` — a pair of small integers, comparable in O(1). After `ceil(log2 n)` rounds the ranks are all distinct and the order is final. Each round sorts in O(n log n), giving **O(n log² n)**.

The optimal **SA-IS** algorithm builds a suffix array in true **O(n)** by an induced-sorting recursion; it's what production libraries use, but it's intricate — we *name* it and stick with the clear prefix-doubling build here.

In [4]:
def suffix_array(s):
    n = len(s)
    sa = list(range(n))
    rank = [ord(c) for c in s]                 # round 0: rank by the first character
    tmp = [0] * n
    k = 1
    while True:
        # order each suffix by (rank of first k chars, rank of the next k chars)
        key = lambda i: (rank[i], rank[i + k] if i + k < n else -1)
        sa.sort(key=key)
        tmp[sa[0]] = 0
        for j in range(1, n):                  # re-rank: equal keys share a rank
            tmp[sa[j]] = tmp[sa[j - 1]] + (key(sa[j]) != key(sa[j - 1]))
        rank = tmp[:]
        if rank[sa[-1]] == n - 1:              # all ranks distinct -> fully sorted
            break
        k <<= 1                                # double the compared length
    return sa

banana = 'banana'
sa = suffix_array(banana)
print('suffix_array("banana"):', sa)
for i in sa:
    print(f'  {i}: {banana[i:]}')             # printed in sorted order

suffix_array("banana"): [5, 3, 1, 0, 4, 2]
  5: a
  3: ana
  1: anana
  0: banana
  4: na
  2: nana


In [5]:
# PROOF: prefix-doubling SA equals the naive sorted-suffix order, on random strings.
def sa_naive(s):
    return sorted(range(len(s)), key=lambda i: s[i:])

for _ in range(2000):
    s = ''.join(rng.choice('ab') for _ in range(rng.randint(1, 12)))
    assert suffix_array(s) == sa_naive(s), s
# include a larger alphabet too
for _ in range(1000):
    s = ''.join(rng.choice('abcd') for _ in range(rng.randint(1, 20)))
    assert suffix_array(s) == sa_naive(s), s
print('suffix_array == sorted(suffixes): 3000 random cases OK')

# Substring search becomes a binary search over the SA band (each compare is O(m)).
import bisect
def sa_contains(s, sa, pattern):
    lo = bisect.bisect_left(range(len(sa)), pattern, key=lambda j: s[sa[j]:sa[j] + len(pattern)])
    return lo < len(sa) and s[sa[lo]:sa[lo] + len(pattern)] == pattern

print('"nan" in banana:', sa_contains(banana, sa, 'nan'))    # True
print('"xyz" in banana:', sa_contains(banana, sa, 'xyz'))    # False

suffix_array == sorted(suffixes): 3000 random cases OK
"nan" in banana: True
"xyz" in banana: False


## 4. LCP array — Kasai's algorithm and the longest repeated substring

The suffix array sorts suffixes; the **LCP array** records how much *adjacent* sorted suffixes overlap:

> `lcp[i]` = length of the longest common prefix of `suffix(sa[i-1])` and `suffix(sa[i])`  (and `lcp[0] = 0`).

**Kasai's algorithm** computes it in **O(n)** using one insight: if suffix `i` shares an LCP of `h` with its sorted predecessor, then suffix `i+1` (one character shorter at the front) shares at least `h-1`. So we walk the original positions `0,1,2,...` and never reset `h` below `h-1` — the LCP pointer advances and retreats by at most 1 per step, giving linear total work. It needs the **inverse** of the suffix array (`rank[i]` = where suffix `i` sits in sorted order) to find each suffix's predecessor.

The payoff: the **longest repeated substring** is exactly the longest LCP entry — two suffixes sharing a `k`-length prefix means that `k`-length string occurs at (at least) two positions.

In [6]:
def lcp_kasai(s, sa):
    n = len(s)
    rank = [0] * n                             # inverse permutation: rank[sa[i]] = i
    for i in range(n):
        rank[sa[i]] = i
    lcp = [0] * n
    h = 0
    for i in range(n):                         # walk ORIGINAL positions, not sorted ones
        if rank[i] > 0:
            j = sa[rank[i] - 1]                # the suffix just before suffix i in sorted order
            while i + h < n and j + h < n and s[i + h] == s[j + h]:
                h += 1
            lcp[rank[i]] = h
            if h:
                h -= 1                          # carry over h-1 to the next original position
        else:
            h = 0
    return lcp

sa = suffix_array(banana)
lcp = lcp_kasai(banana, sa)
print('suffix_array("banana"):', sa)
print('lcp_kasai  ("banana"):', lcp)
for i in range(len(sa)):
    print(f'  lcp={lcp[i]}  {banana[sa[i]:]}')

suffix_array("banana"): [5, 3, 1, 0, 4, 2]
lcp_kasai  ("banana"): [0, 1, 3, 0, 0, 2]
  lcp=0  a
  lcp=1  ana
  lcp=3  anana
  lcp=0  banana
  lcp=0  na
  lcp=2  nana


In [7]:
def longest_repeated_substring(s):
    if len(s) < 2:
        return ''
    sa = suffix_array(s)
    lcp = lcp_kasai(s, sa)
    i = max(range(len(s)), key=lambda k: lcp[k])
    return s[sa[i]: sa[i] + lcp[i]]

print('longest repeat in "mississippi":', repr(longest_repeated_substring('mississippi')))
print('longest repeat in "banana"     :', repr(longest_repeated_substring('banana')))

# PROOF: LCP matches a direct pairwise computation, and the longest-repeat length
#        matches an O(n^2) substring-positions brute force.
def lcp_brute(s, sa):
    out = [0] * len(s)
    for i in range(1, len(s)):
        a, b, k = sa[i - 1], sa[i], 0
        while a + k < len(s) and b + k < len(s) and s[a + k] == s[b + k]:
            k += 1
        out[i] = k
    return out

def lrs_len_brute(s):                          # longest substring occurring at >= 2 positions
    from collections import defaultdict
    pos = defaultdict(set)
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            pos[s[i:j]].add(i)
    return max((len(sub) for sub, p in pos.items() if len(p) >= 2), default=0)

for _ in range(2000):
    s = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 12)))
    sa = suffix_array(s)
    assert lcp_kasai(s, sa) == lcp_brute(s, sa), s
    assert len(longest_repeated_substring(s)) == lrs_len_brute(s), s
print('lcp_kasai == pairwise LCP, and LRS length == brute force: 2000 random cases OK')

longest repeat in "mississippi": 'issi'
longest repeat in "banana"     : 'ana'
lcp_kasai == pairwise LCP, and LRS length == brute force: 2000 random cases OK


## 5. Aho-Corasick — many patterns in one O(|T|) scan

To search for *many* patterns at once, build a **trie** of all of them (see the **tries** notebook), then add **failure links** so the automaton can recover from a mismatch without rescanning the text — exactly **KMP**'s prefix-function idea, generalized from one string to a whole trie. `fail[v]` points to the node spelling the **longest proper suffix** of `v`'s string that is itself a node in the trie; failure links are filled level by level with a **BFS** (a node's fail target is always shallower, so it's already computed).

Two more pieces make it linear:

- **Output links:** a node's match list is *merged* with its fail target's, so when several patterns end at the same point (e.g. `he` ending inside `she`) one node reports them all.
- **The scan:** read the text once; on a mismatch follow `fail` until a transition exists (or we hit the root). Total time **O(|T| + total matches)**, independent of how many patterns there are.

We store the trie as parallel arrays (`goto`, `fail`, `out`) indexed by node id — a compact, cache-friendly layout.

In [8]:
class AhoCorasick:
    def __init__(self, patterns):
        self.patterns = list(patterns)
        self.goto = [{}]          # goto[v][ch] -> child node id   (the trie edges)
        self.fail = [0]           # fail[v] -> longest-proper-suffix node
        self.out = [[]]           # out[v]  -> pattern indices reported at v
        for idx, p in enumerate(self.patterns):
            v = 0
            for ch in p:          # insert like any trie (cf. the tries notebook)
                if ch not in self.goto[v]:
                    self.goto.append({}); self.fail.append(0); self.out.append([])
                    self.goto[v][ch] = len(self.goto) - 1
                v = self.goto[v][ch]
            self.out[v].append(idx)
        self._build_links()

    def _build_links(self):
        q = deque(self.goto[0].values())          # depth-1 nodes: fail link is the root (0)
        while q:
            v = q.popleft()
            for ch, nxt in self.goto[v].items():
                f = self.fail[v]
                while f and ch not in self.goto[f]:
                    f = self.fail[f]              # walk fail links (KMP-style) until ch fits
                self.fail[nxt] = self.goto[f].get(ch, 0)
                self.out[nxt] += self.out[self.fail[nxt]]   # inherit the output link
                q.append(nxt)

    def search(self, text):
        """Yield (start_index, pattern_index) for every occurrence."""
        v = 0
        for i, ch in enumerate(text):
            while v and ch not in self.goto[v]:
                v = self.fail[v]                  # mismatch -> fall back, never rescan text
            v = self.goto[v].get(ch, 0)
            for idx in self.out[v]:
                yield (i - len(self.patterns[idx]) + 1, idx)

ac = AhoCorasick(['he', 'she', 'his', 'hers'])
for start, idx in sorted(ac.search('ushers')):
    print(f'  pos {start}: {ac.patterns[idx]!r}')     # finds she, he, hers in one scan

  pos 1: 'she'
  pos 2: 'he'
  pos 2: 'hers'


In [9]:
# PROOF: Aho-Corasick finds exactly the same occurrences as searching each pattern alone.
def brute_multi(text, patterns):
    out = []
    for idx, p in enumerate(patterns):
        if p:
            out += [(i, idx) for i in brute_search(text, p)]
    return sorted(out)

for _ in range(4000):
    k = rng.randint(1, 5)
    patterns = [''.join(rng.choice('abc') for _ in range(rng.randint(1, 5))) for _ in range(k)]
    text = ''.join(rng.choice('abc') for _ in range(rng.randint(0, 22)))
    got = sorted(AhoCorasick(patterns).search(text))
    assert got == brute_multi(text, patterns), (patterns, text)
print('Aho-Corasick == per-pattern brute force: 4000 random multi-pattern cases OK')

Aho-Corasick == per-pattern brute force: 4000 random multi-pattern cases OK


**Why not just loop KMP per pattern?** That costs `O(k·|T|)` for `k` patterns — Aho-Corasick collapses it to a single `O(|T|)` pass plus the matches, because the shared trie scans all patterns simultaneously. With thousands of patterns (virus signatures, banned words, dictionary terms) the difference is decisive. The price is `O(total pattern length)` memory for the automaton.

## 6. When to use what

| Situation | Reach for | Why |
|---|---|---|
| One pattern, one text | **KMP** / Rabin-Karp (string algorithms) | O(n+m), no preprocessing of the text |
| Pattern matching, need a clean linear primitive | **Z-algorithm** | one array, `P + sep + T`; also gives string periods |
| **Many substring queries on a fixed text** | **suffix array** | sort suffixes once, then binary-search any query |
| Longest repeat / distinct substrings / overlaps | **suffix array + LCP** | LCP turns repeats into adjacent-suffix overlaps |
| Production suffix array on huge text | **SA-IS** (named, not built here) | true O(n) induced sorting |
| **Many patterns, one scan** | **Aho-Corasick** | O(|T| + matches), independent of pattern count |
| Just `pattern in text` in Python | built-in `in` / `.find` | C `fastsearch` beats any pure-Python build |

Rule of thumb: **preprocess the text** (suffix array/LCP) when the *text* is reused across many queries; **preprocess the patterns** (Aho-Corasick) when the *pattern set* is reused across many texts/streams.

## 7. Complexity cheat-sheet

<sub>n = text length, m = pattern length, k = number of patterns, M = total pattern length, z = number of matches.</sub>

| Structure | Build | Query | Space | Notes |
|---|:---:|:---:|:---:|---|
| Z-array | O(n) | O(n) per `P+sep+T` | O(n) | prefix-match lengths; KMP-cousin |
| Suffix array (prefix-doubling) | O(n log² n) | O(m log n) substring search | O(n) | sorted suffixes; binary-searchable |
| Suffix array (SA-IS) | O(n) | O(m log n) | O(n) | optimal build, used in practice |
| LCP array (Kasai) | O(n) given the SA | O(1) lookup | O(n) | longest-repeat = max LCP |
| Aho-Corasick | O(M) | O(n + z) scan | O(M·Σ) | multi-pattern; KMP over a trie |
| Suffix tree (named only) | O(n) | O(m) substring | O(n) | strongest, heaviest constants |

<sub>Σ = alphabet size (the per-node `dict` cost). In Python you reach for these only when the built-ins can't: reused-text indexes, longest-repeat/dedup, and thousands-of-patterns scanning. For a plain `pattern in text`, the C `in`/`.find` of the **string algorithms** notebook still wins.</sub>